<div style="border-top: 3px solid #ac543f; padding-top: 12px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Technical assessment &middot; 30-minute presentation</span></div>

# <span style="font-family: Newsreader, Georgia, serif; font-size: 48px; font-weight: 400;">Concept Transformer on CUB-200-2011</span>

<div style="font-family: Newsreader, Georgia, serif; font-style: italic; font-size: 20px; line-height: 1.4; opacity: 0.85; max-width: 44em;">Reproducing &ldquo;Attention-based Interpretability with Concept Transformers&rdquo; (Rigotti et al., ICLR 2022) and reading what its concepts explain</div>
<div style="font-size: 13.5px; opacity: 0.65; margin-top: 10px;">Presenter name &middot; October 2026 &middot; HiPerGator, one NVIDIA L4</div>
<div style="margin-top: 26px;"><div style="display: inline-block; vertical-align: top; padding-top: 8px; width: 23%; margin-right: 2%; border-top: 1px solid rgba(128, 128, 128, 0.45);"><div style="font-family: Newsreader, Georgia, serif; font-size: 28px; font-weight: 300; line-height: 1.2;">5 min</div><div style="font-size: 13.5px; line-height: 1.4; opacity: 0.8;">Paper and model overview</div></div><div style="display: inline-block; vertical-align: top; padding-top: 8px; width: 23%; margin-right: 2%; border-top: 1px solid rgba(128, 128, 128, 0.45);"><div style="font-family: Newsreader, Georgia, serif; font-size: 28px; font-weight: 300; line-height: 1.2;">10 min</div><div style="font-size: 13.5px; line-height: 1.4; opacity: 0.8;">Walkthrough: setup, data, training, results</div></div><div style="display: inline-block; vertical-align: top; padding-top: 8px; width: 23%; margin-right: 2%; border-top: 1px solid rgba(128, 128, 128, 0.45);"><div style="font-family: Newsreader, Georgia, serif; font-size: 28px; font-weight: 300; line-height: 1.2;">10 min</div><div style="font-size: 13.5px; line-height: 1.4; opacity: 0.8;">Live demonstration</div></div><div style="display: inline-block; vertical-align: top; padding-top: 8px; width: 23%; border-top: 1px solid rgba(128, 128, 128, 0.45);"><div style="font-family: Newsreader, Georgia, serif; font-size: 28px; font-weight: 300; line-height: 1.2;">5 min</div><div style="font-size: 13.5px; line-height: 1.4; opacity: 0.8;">Discussion and next steps</div></div></div>

<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Configuration &middot; run first</span></div>
<div style="font-size: 15px; line-height: 1.5; margin-top: 6px; max-width: 46em;">The paths below are the only settings to edit. Every number from our runs is read from the verified records saved with <code>concept_transformer_walkthrough.ipynb</code>.</div>

In [ ]:
# Configuration: the only paths to edit. Run this cell first; every section relies on these names.
from pathlib import Path
import json
import os
import platform
import sys

PROJECT_DIR = Path.cwd()  # repository root: the folder holding this notebook
WORK_DIR = Path("/blue/yonghui.wu/zhangm3/concept_transformer_explain")  # HiPerGator working copy with data and venv
DATA_DIR = WORK_DIR / "CUB-200-2011" / "CUB_200_2011"  # the --data_dir passed to cvit_cub.py
REPO_DIR = PROJECT_DIR / "concept_transformer"  # IBM repository code at the pinned commit
WALKTHROUGH = PROJECT_DIR / "concept_transformer_walkthrough.ipynb"  # executed notebook holding the run records
ARTIFACT_DIR = PROJECT_DIR / "explanation_artifacts" / "disjoint-45478721"
SETUP_LOGS = PROJECT_DIR / "setup_logs"

assert (REPO_DIR / "cvit_cub.py").is_file(), "Set PROJECT_DIR to the repository folder."
assert sys.version_info[:2] == (3, 8), "Select the 'Concept Transformer (HPG Python 3.8)' kernel."
assert platform.system() == "Linux", "Run in the HiPerGator Linux environment."

# Cap BLAS threads at the Slurm allocation before numpy or torch start their thread pools.
CPU_THREADS = max(1, min(int(os.environ.get("SLURM_CPUS_PER_TASK", "4")), len(os.sched_getaffinity(0))))
for variable in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[variable] = str(CPU_THREADS)
os.environ["TORCH_HOME"] = str(PROJECT_DIR / ".cache" / "torch")
os.environ["MPLCONFIGDIR"] = str(PROJECT_DIR / ".cache" / "matplotlib")
for path in (PROJECT_DIR, REPO_DIR):  # this notebook's helpers, then the repository's ctc/ and data/
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display
from presentation.helpers import PALETTE, caption, fact_table, show_image, stat_row, use_paper_style

use_paper_style()
EVIDENCE = json.loads(WALKTHROUGH.read_text())["metadata"]  # verified records of both completed L4 runs
RUNS = EVIDENCE["completed_run_comparison"]["runs"]
REPO_RUN, DISJOINT_RUN = "45387524", "45478721"
RUN_LABEL = {REPO_RUN: "Repository split", DISJOINT_RUN: "Disjoint split"}
RUN_COLOR = {REPO_RUN: PALETTE["neg"], DISJOINT_RUN: PALETTE["accent"]}
assert all(RUNS[job]["snapshot"]["run_status"] == "complete" for job in (REPO_RUN, DISJOINT_RUN))
print(f"Ready: {PROJECT_DIR} | Python {platform.python_version()} | {CPU_THREADS} CPU threads")